In [ ]:
##1. Imports

#import functions

import librosa
import numpy as np
import matplotlib.pyplot as plt
import scipy
import os

In [ ]:
##2. Load Audio [INSERT FILE PATH]

##Shorthands for commonly used files
A_harmonic_minor_path = "Audios/Chord Audios/Standardized/A Harmonic Minor Am - Dm - E - Am - F - G#dim - E - Am.wav"
A_natural_minor_path = "Audios/Chord Audios/Standardized/A Natural Minor (Aeolian) Am - F - C - G - Am - Dm - Em - Am.wav"
B_locrian_path = "Audios/Chord Audios/Standardized/B Locrian Bdim - C - Dm - C - Bdim - C - Dm - Em.wav"
C_major_path = "Audios/Chord Audios/Standardized/C Major (Ionian) C - Am - F - G - Em - F - G - C.wav"
D_dorian_path = "Audios/Chord Audios/Standardized/D Dorian Dm - F - Am - G - Dm - Em - F - Em.wav"
E_mixolydian_path = "Audios/Chord Audios/Standardized/E Mixolydian E - D - A - E - E - F#m - Bm - E.wav"
G_phrygian_path = "Audios/Chord Audios/Standardized/G Phrygian Gm - Ab - Gm - Fm - Gm - Cm - Bb - Ab.wav"
E_phrygian_dominant_path = "Audios/Chord Audios/Standardized/E Phrygian Dominant E - F - E - F - Am - Dm - F - E.wav"
F_lydian_path = "Audios/Chord Audios/Standardized/F Lydian F - G - Am - G - F - Am - G_B - C.wav"


audio_path = A_harmonic_minor_path # Change this to the desired audio file path
y, sr = librosa.load(audio_path, sr=None)  # Load audio file, sr=None to preserve original sampling rate
filename = os.path.basename(audio_path)

#Note: y --> audio array, sr --> sampling rate
#Note to self: May see Py SoundFile failed. Trying audioread instead --> warning message, not an error

In [ ]:
##3. Tempo Detection

#In theory, BPM could be determined by detecting peaks in the audio signal, which correspond to beats. 
#I will start with a simple approach using the librosa library.

#Beat Detection
tempo, beat_frames = librosa.beat.beat_track(y=y, sr=sr)
print(f"Estimated Tempo: {tempo} BPM")

#Alternative calcuation
beat_times = librosa.frames_to_time(beat_frames, sr=sr)
beat_intervals = np.diff(beat_times)

bpm_from_intervals = 60 / np.mean(beat_intervals)
print(f"BPM from intervals: [{bpm_from_intervals}] BPM")

In [ ]:
##4 Tempo Curve [EXPERIMENTAL]
onset_env = librosa.onset.onset_strength(y=y, sr=sr)
dtempo = librosa.feature.tempo(onset_envelope=onset_env, sr=sr, aggregate=None)

plt.figure(figsize=(12, 4))
plt.plot(librosa.times_like(dtempo), dtempo)
plt.title("Tempo over Time")
plt.xlabel("Time (s)")
plt.ylabel("BPM")
plt.show()

from scipy.ndimage import uniform_filter1d

dtempo_smooth = uniform_filter1d(dtempo, size=2000) # Adjust size for more or less smoothing

plt.figure(figsize=(12, 4))
plt.plot(librosa.times_like(dtempo), dtempo_smooth)
plt.title("Tempo over Time (Smoothed)")
plt.xlabel("Time (s)")
plt.ylabel("BPM")
plt.show()

In [ ]:
##5. Chromagram 
chromagram = librosa.feature.chroma_cqt(y=y, sr=sr)

#Note: We use cqt and not stft as cqt is logarithmic, which corresponds better to human perception of pitch.

print(chromagram.shape)
#12 --> corresponds with the 12 notes
#The second number is the number of time frames: the number of samplings/snapshots

tuning = librosa.estimate_tuning(y=y, sr=sr)
print(f"Tuning offset: {tuning} cents")

In [ ]:
##5A: Chromogram Heatmap
#Great for visualizing how the intensity of each pitch changes over time.
fig = plt.figure()
librosa.display.specshow(chromagram, sr=sr, x_axis = "time", y_axis = "chroma")
plt.colorbar()
plt.title("Chromagram Heatmap")
plt.show()

In [ ]:
##5B: Chromagram Barchart
#Great for visualiziing which pitches are most prominent in the audio.
chroma_mean = np.mean(chromagram, axis=1)
#Note: np.mean with axis=1 calculates the mean across the time frames for each of the 12 bins
pitch_names = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
plt.bar(pitch_names, chroma_mean)
plt.xlabel("Note")
plt.ylabel("Mean Intensity")
plt.title("Chromagram Barchart")
plt.show()

In [ ]:
##6. Key Detection (music21) 
from music21 import stream, key, note

from music21.analysis.discrete import KrumhanslSchmuckler
analyzer = KrumhanslSchmuckler()

s = stream.Stream()

for pitch_name, value in zip(pitch_names, chroma_mean):
    i = int(value * 100)
    while i > 0:
        n = note.Note(pitch_name)
        s.append(n)
        i -= 1


result = analyzer.getSolution(s)
print(f"Estimated Key: {result}")


#Can use following for alternative estimates:
for alt in result.alternateInterpretations[:4]:
    print(f"Alternative: {alt}")



# Note: the estimation is not exactly accurate all the time, especially for minor keys 
# where KrumhanslSchmuckler doesn't account for harmonics

In [ ]:
##7. Mode Detection (Custom)

#My OWN implementation of a simple key estimation based on the chroma mean and the Krumhansl-Schmuckler profiles.
#Note: This is a very basic implementation and may not be as accurate as the one provided
#by music21, but it serves as a good exercise in understanding the key estimation process.



#Update 1: Testing using 8.0 for tonic rather than 6.0
#Yields better results for songs with strong tonic presence, but may yield worse results for songs with weak tonic presence.

#Default: 6.0 for 1st, 3rd, 5th, 3.0 for notes in scale, 1.0 for rest)

#Test 1: Testing using 8.0 for tonic rather than 6.0
#Yields better results for songs with strong tonic presence, but may yield worse results for songs with weak tonic presence.

#Test 2: Testing changing the 1.0 to 0.2
#Did not necessarily yield better results, helped some but hurt others (i.e. Phrygian Dominant got better, but Dorian got worse)


c_lydian = [8.0, 1.0, 3.0, 1.0, 6.0, 1.0, 3.0, 6.0, 1.0, 3.0, 1.0, 3.0]
#             C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_major = [8.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 6.0, 1.0, 3.0, 1.0, 3.0]
#           C    C#   D    D#   E    F    F#   G    G#   A    A#   B

#Note: May need some reweighting as the scale gets flagged as phrygian as well
c_mixolydian = [8.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 6.0, 1.0, 3.0, 3.0, 1.0]
#                C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_dorian = [8.0, 1.0, 3.0, 6.0, 1.0, 3.0, 1.0, 6.0, 1.0, 3.0, 3.0, 1.0]
#            C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_natural_minor = [8.0, 1.0, 3.0, 6.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 3.0, 1.0]
#                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B  

c_harmonic_minor = [8.0, 1.0, 3.0, 6.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 1.0, 3.0]
#                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_phrygian = [8.0, 3.0, 1.0, 6.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 3.0, 1.0]
#             C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_phrygian_dominant = [8.0, 3.0, 1.0, 1.0, 6.0, 3.0, 1.0, 6.0, 3.0, 1.0, 3.0, 1.0]
#                       C    C#   D    D#   E    F    F#   G    G#   A    A#   B

#Note: F# could be moved from 6.0 to 3.0 weight, yet to be determined
c_locrian = [8.0, 3.0, 1.0, 6.0, 1.0, 3.0, 6.0, 1.0, 3.0, 1.0, 3.0, 1.0]
#             C    C#   D    D#   E    F    F#   G    G#   A    A#   B




harmonic_minor_profiles = []
for i in range(12):
    harmonic_minor_profiles.append(np.roll(c_harmonic_minor, i))

major_profiles = []
for i in range(12):
    major_profiles.append(np.roll(c_major, i))

minor_profiles = []
for i in range(12):
    minor_profiles.append(np.roll(c_natural_minor, i))

dorian_profiles = []
for i in range(12):
    dorian_profiles.append(np.roll(c_dorian, i))

phrygian_profiles = []
for i in range(12):
    phrygian_profiles.append(np.roll(c_phrygian, i))

locrian_profiles = []
for i in range(12):
    locrian_profiles.append(np.roll(c_locrian, i))

lydian_profiles = []
for i in range(12):
    lydian_profiles.append(np.roll(c_lydian, i))

mixolydian_profiles = []
for i in range(12):
    mixolydian_profiles.append(np.roll(c_mixolydian, i))

phrygian_dominant_profiles = []
for i in range(12):
    phrygian_dominant_profiles.append(np.roll(c_phrygian_dominant, i))


major_pitch_names = ['C Major', 'Db Major', 'D Major', 'Eb Major', 'E Major', 'F Major', 'F# Major', 'G Major', 'Ab Major', 'A Major', 'Bb Major', 'B Major']
minor_pitch_names = ['C Minor', 'C# Minor', 'D Minor', 'Eb Minor', 'E Minor', 'F Minor', 'F# Minor', 'G Minor', 'G# Minor', 'A Minor', 'Bb Minor', 'B Minor']
harmonic_pitch_names = ['C Harmonic Minor', 'C# Harmonic Minor', 'D Harmonic Minor', 'Eb Harmonic Minor', 'E Harmonic Minor', 'F Harmonic Minor', 'F# Harmonic Minor', 'G Harmonic Minor', 'G# Harmonic Minor', 'A Harmonic Minor', 'Bb Harmonic Minor', 'B Harmonic Minor']
dorian_pitch_names = ['C Dorian', 'C# Dorian', 'D Dorian', 'Eb Dorian', 'E Dorian', 'F Dorian', 'F# Dorian', 'G Dorian', 'Ab Dorian', 'A Dorian', 'Bb Dorian', 'B Dorian']
phrygian_pitch_names = ['C Phrygian', 'C# Phrygian', 'D Phrygian', 'D# Phrygian', 'E Phrygian', 'F Phrygian', 'F# Phrygian', 'G Phrygian', 'G# Phrygian', 'A Phrygian', 'Bb Phrygian', 'B Phrygian']
locrian_pitch_names = ['C Locrian', 'C# Locrian', 'D Locrian', 'D# Locrian', 'E Locrian', 'F Locrian', 'F# Locrian', 'G Locrian', 'G# Locrian', 'A Locrian', 'Bb Locrian', 'B Locrian']
lydian_pitch_names = ['C Lydian', 'Db Lydian', 'D Lydian', 'Eb Lydian', 'E Lydian', 'F Lydian', 'F# Lydian', 'G Lydian', 'Ab Lydian', 'A Lydian', 'Bb Lydian', 'B Lydian']
mixolydian_pitch_names = ['C Mixolydian', 'C# Mixolydian', 'D Mixolydian', 'Eb Mixolydian', 'E Mixolydian', 'F Mixolydian', 'F# Mixolydian', 'G Mixolydian', 'Ab Mixolydian', 'A Mixolydian', 'Bb Mixolydian', 'B Mixolydian']
phrygian_dominant_pitch_names = ['C Phrygian Dominant', 'C# Phrygian Dominant', 'D Phrygian Dominant', 'D# Phrygian Dominant', 'E Phrygian Dominant', 'F Phrygian Dominant', 'F# Phrygian Dominant', 'G Phrygian Dominant', 'G# Phrygian Dominant', 'A Phrygian Dominant', 'Bb Phrygian Dominant', 'B Phrygian Dominant']   


All_names = major_pitch_names + minor_pitch_names + harmonic_pitch_names + dorian_pitch_names + phrygian_pitch_names + locrian_pitch_names + lydian_pitch_names + mixolydian_pitch_names + phrygian_dominant_pitch_names
scores = []
for profile in major_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in minor_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in harmonic_minor_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in dorian_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in phrygian_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in locrian_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in lydian_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in mixolydian_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])
for profile in phrygian_dominant_profiles:
    score = np.corrcoef(chroma_mean, profile)
    scores.append(score[0, 1])

top3_indices = np.argsort(scores)[-3:][::-1]
for i in top3_indices:
    print(f"{All_names[i]}: {scores[i]:.4f}")

In [ ]:
##8. Parent Key Indentification

#Step 1: Identify 7 most prominent pitches
top_7 = set(np.argsort(chroma_mean)[-7:])  # Get indices of top 7 pitches
print("Top 7 Pitches (0=C, 1=C#, ..., 11=B):", top_7)

#step 2: Major scale profiles
major_scales = {
    'C':  {0, 2, 4, 5, 7, 9, 11},
    'Db': {1, 3, 5, 6, 8, 10, 0},
    'D':  {2, 4, 6, 7, 9, 11, 1},
    'Eb': {3, 5, 7, 8, 10, 0, 2},
    'E':  {4, 6, 8, 9, 11, 1, 3},
    'F':  {5, 7, 9, 10, 0, 2, 4},
    'F#': {6, 8, 10, 11, 1, 3, 5},
    'G':  {7, 9, 11, 0, 2, 4, 6},
    'Ab': {8, 10, 0, 1, 3, 5, 7},
    'A':  {9, 11, 1, 2, 4, 6, 8},
    'Bb': {10, 0, 2, 3, 5, 7, 9},
    'B':  {11, 1, 3, 4, 6, 8, 10},
}


parent_key_scores = {}
for key_name, scale_notes in major_scales.items():
    parent_key_scores[key_name] = sum(chroma_mean[i] for i in scale_notes)

top3_keys = sorted(parent_key_scores.items(), key=lambda x: x[1], reverse=True)[:3]
print("Top 3 Parent Keys:")
for key, score in top3_keys:
    print(f"  {key}: {score:.4f}")

In [ ]:
##9. Modal Alignment & Coherence Check


#Add the offsets [# of semitones]
# pythonmode_to_parent_offset = {
#     'Major': 0,
#     'Dorian': 2,
#     'Phrygian': 4,
#     'Phrygian Dominant': 4,
#     'Lydian': 5,
#     'Mixolydian': 7,
#     'Minor': 9,
#     'Harmonic Minor': 9,
#     'Locrian': 11
    
# }

#The actual correct one; turns out I need to take 12 - semitones for the way parent index is calculated
mode_to_parent_offset = {
    'Major': 0,
    'Dorian': 10,
    'Phrygian': 8,
    'Phrygian Dominant': 8,
    'Lydian': 7,
    'Mixolydian': 5,
    'Minor': 3,
    'Harmonic Minor': 3,
    'Locrian': 1
}

mode_result = top3_indices


pitch_names = ['C', 'Db', 'D', 'Eb', 'E', 'F', 'F#', 'G', 'Ab', 'A', 'Bb', 'B']

FLAT_TO_SHARP = {
    'C#': 'Db',
    'D#': 'Eb',
    'Gb': 'F#',
    'G#': 'Ab',
    'A#': 'Bb'
}

SHARP_TO_FLAT = {v: k for k, v in FLAT_TO_SHARP.items()}

def notes_are_equivalent(note1, note2):
    """Returns True if notes sound identical (e.g., A# and Bb)."""
    n1_sharp = FLAT_TO_SHARP.get(note1, note1)
    n2_sharp = FLAT_TO_SHARP.get(note2, note2)
    return n1_sharp == n2_sharp

def get_parent_key(mode_result):
    # Split "D Mixolydian" into tonic and mode type
    parts = mode_result.split(' ', 1)
    tonic = parts[0]
    mode_type = parts[1]
    
    if tonic in FLAT_TO_SHARP:
        tonic = FLAT_TO_SHARP[tonic]

    # Get tonic index
    tonic_index = pitch_names.index(tonic)
    
    # Get offset
    offset = mode_to_parent_offset.get(mode_type, 0)
    
    # Calculate parent key index
    parent_index = (tonic_index + offset) % 12
    
    return pitch_names[parent_index]

print("=== Modal Alignment & Coherence Check ===")
print("Verifying that predicted modes resolve to the detected parent key signatures:\n")
print("Cross Validation:")
for i in top3_indices:
    mode_result = All_names[i]
    parent = get_parent_key(mode_result) # e.g., returns "A#"
    
    # Check if ANY key in top3_keys is an enharmonic match for our parent note
    is_match = False
    for k, _ in top3_keys:
        if notes_are_equivalent(parent, k):
            is_match = True
            break
            
    match = "✅" if is_match else "❌"
    
    # Clean up output string so humans see 'Bb' instead of 'A#' if the target uses flats
    # We grab the first key in top3_keys to check if the dataset prefers flats ('b')
    sample_key = top3_keys[0][0] if top3_keys else ""
    if "b" in sample_key:
        display_parent = SHARP_TO_FLAT.get(parent, parent)
    else:
        display_parent = parent

    print(f"  {mode_result} → Parent Key: {display_parent} {match}")

In [ ]:
##10. Summary Cell
print(f"File: {filename}")

print("=" * 40)
print("AUDIO ANALYSIS SUMMARY")
print("=" * 40)
print(f"Estimated Tempo: {tempo} BPM")
print(f"Tempo from intervals: {bpm_from_intervals:.2f} BPM")
print()
print(f"Krumhansl-Schmuckler Key: {result}")
for alt in result.alternateInterpretations[:4]:
    print(f"  Alternative: {alt}")
print()
print("Custom Mode Estimation:")
for i in top3_indices:
    print(f"{All_names[i]}: {scores[i]:.4f}")
print("=" * 40)



In [ ]:
##11: Confidence Cell

#Note: uses get_parent_key which is defined elsewhere in the notebook to determine the parent key of the mode


top_score = scores[top3_indices[0]]
top_mode = All_names[top3_indices[0]]
top_parent = get_parent_key(top_mode)

top_parent_keys = [k for k, _ in top3_keys]



enharmonic = {
    'Db': 'C#', 'Eb': 'D#', 'Fb': 'E', 'Gb': 'F#', 
    'Ab': 'G#', 'Bb': 'A#', 'Cb': 'B'
}

def normalize_key(key):
    return enharmonic.get(key, key)


if top_score > 0.7:
    print(f"Confidence: HIGH ({top_score:.4f})")
    print(f"Mode: {top_mode}")
    print(f"Parent Key: {top_parent}")
elif top_score > 0.5:
    print(f"Confidence: AMBIGUOUS ({top_score:.4f})")
    print(f"Most likely parent key: {top_parent}")
    print("Possible modes (cross validated):")
    for i in top3_indices:
        mode = All_names[i]
        parent = get_parent_key(mode)

        if normalize_key(parent) in [normalize_key(k) for k in top_parent_keys]:
            print(f"  {mode} → Parent Key: {parent}")
else:
    print(f"Confidence: LOW ({top_score:.4f})")
    print("Tonal center unclear — parent key estimate only:")
    for key, score in top3_keys:
        print(f"  {key}: {score:.4f}")

        

In [ ]:
##Extra: Time Signature [Experimental]

# pulse = librosa.beat.plp(y=y, sr=sr)
# pulse_times = librosa.frames_to_time(np.arange(len(pulse)), sr=sr)
# plt.figure(figsize=(12, 4))
# plt.plot(pulse_times, pulse)
# plt.xlim(0, 5)
# plt.title("Pulse Strength - First 5 seconds")
# plt.xlabel("Time (s)")
# plt.ylabel("Pulse Strength")
# plt.show()

#Note: This one is very difficult, may not use or implement for now.


In [ ]:
# #EXTRA: Detecting Harmonic Minors [Origional Code]

# c_harmonic_minor = [6.0, 1.0, 3.0, 3.0, 1.0, 3.0, 1.0, 6.0, 3.0, 1.0, 1.0, 3.0]
# #                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B
# #Mirrors the weights for the minor scale, but with the raised 7th (harmonic minor) instead of the natural minor.

# harmonic_minor_profiles = []
# for i in range(12):
#     harmonic_minor_profiles.append(np.roll(c_harmonic_minor, i))

# scores = []
# for profile in harmonic_minor_profiles:
#     score = np.corrcoef(chroma_mean, profile)
#     scores.append(score[0, 1])


# harmonic_pitch_names = ['C Harmonic Minor', 'C# Harmonic Minor', 'D Harmonic Minor', 'D# Harmonic Minor', 'E Harmonic Minor', 'F Harmonic Minor', 'F# Harmonic Minor', 'G Harmonic Minor', 'G# Harmonic Minor', 'A Harmonic Minor', 'A# Harmonic Minor', 'B Harmonic Minor']

# highest_score_index = np.argmax(scores)
# estimated_key_harmonic_minor = harmonic_pitch_names[highest_score_index]
# print(f"Estimated Key (Harmonic Minor): {estimated_key_harmonic_minor}")

In [ ]:
##Copy Paste Code


#Test weightings: 8.0, 0.2

c_lydian = [8.0, 0.2, 3.0, 0.2, 6.0, 0.2, 3.0, 6.0, 0.2, 3.0, 0.2, 3.0]
#             C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_major = [8.0, 0.2, 3.0, 0.2, 6.0, 3.0, 0.2, 6.0, 0.2, 3.0, 0.2, 3.0]
#           C    C#   D    D#   E    F    F#   G    G#   A    A#   B

#Note: May need some reweighting as the scale gets flagged as phrygian as well
c_mixolydian = [8.0, 0.2, 3.0, 0.2, 6.0, 3.0, 0.2, 6.0, 0.2, 3.0, 3.0, 0.2]
#                C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_dorian = [8.0, 0.2, 3.0, 6.0, 0.2, 3.0, 0.2, 6.0, 0.2, 3.0, 3.0, 0.2]
#            C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_natural_minor = [8.0, 0.2, 3.0, 6.0, 0.2, 3.0, 0.2, 6.0, 3.0, 0.2, 3.0, 0.2]
#                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B  

c_harmonic_minor = [8.0, 0.2, 3.0, 6.0, 0.2, 3.0, 0.2, 6.0, 3.0, 0.2, 0.2, 3.0]
#                   C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_phrygian = [8.0, 3.0, 0.2, 6.0, 0.2, 3.0, 0.2, 6.0, 3.0, 0.2, 3.0, 0.2]
#             C    C#   D    D#   E    F    F#   G    G#   A    A#   B

c_phrygian_dominant = [8.0, 3.0, 0.2, 0.2, 6.0, 3.0, 0.2, 6.0, 3.0, 0.2, 3.0, 0.2]
#                       C    C#   D    D#   E    F    F#   G    G#   A    A#   B

#Note: F# could be moved from 6.0 to 3.0 weight, yet to be determined
c_locrian = [8.0, 3.0, 0.2, 6.0, 0.2, 3.0, 6.0, 0.2, 3.0, 0.2, 3.0, 0.2]
#             C    C#   D    D#   E    F    F#   G    G#   A    A#   B